# Cobolt Laser Blink Routine

Drive a Cobolt laser through `pycobolt`: set power, toggle emission ON/OFF on an
absolute clock schedule, watch the cycles run, and log the timing errors to a CSV.

> **Safety first.** 808 nm is invisible. Close the interlock, wear the rated
> goggles, keep the beam path enclosed, and confirm the COM port **before** you
> enable emission. This notebook turns the laser on.

> **What is actually measured.** Every number here is a *software* timing: how
> late Python issued the serial command, and how long the command took to return.
> It is **not** the optical ON/OFF duration — the diode has its own rise/fall and
> thermal behaviour. Validate with a photodiode + DAQ/scope before trusting a
> protocol. Serial toggling is good to ~10 ms; for sub-ms precision use the
> laser's **modulation input** (see §7).

**Workflow:** §1–§4 once, then re-run the cells in **§5** as often as you like —
each `run(...)` call is self-contained, writes its own timestamped CSV and is
remembered in the §6 comparison table.

## 1  Defaults and hardware limits
Everything here is optional to repeat: `run()` in §5 takes per-run values and falls back to these.

In [ ]:
import os

COM_PORT = os.environ.get("COBOLT_PORT", "COM8")  # Windows "COM8" · Linux "/dev/ttyUSB0"
MAX_POWER_MW = 100.0  # rated max of THIS head - pycobolt cannot query it, so we clamp

# Defaults used by run() for any argument you leave out
POWER_MW = 100.0    # output power, mW
ON_S     = 0.05     # ON dwell, s
OFF_S    = 0.05     # OFF dwell, s
N_CYCLES = 200
SPIN_S   = 0.001    # busy-wait window at the end of each sleep, s (0 = pure sleep)
PROGRESS = True     # live progress bar per cycle
LOG_DIR  = "logs"   # one timestamped CSV per run; nothing is ever overwritten

## 2  Helpers

In [ ]:
import contextlib, csv, pathlib, sys, time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pycobolt

BAR_MIN_PERIOD_S = 0.02   # below this, a per-cycle UI update would eat the timing budget


def pkg_version(name):
    """Installed version of a package, or 'unknown' (pycobolt defines no __version__)."""
    try:
        from importlib.metadata import version
        return version(name)
    except Exception:
        return "unknown"


def call_if_available(obj, name, *args):
    """Call obj.name(*args) if that method exists, else None.

    pycobolt exposes a different command set per model/firmware, so optional
    queries are probed instead of assumed.
    """
    fn = getattr(obj, name, None)
    return fn(*args) if callable(fn) else None


def wait_until(deadline, spin_s):
    """Sleep most of the way to `deadline`, then busy-wait the last `spin_s`."""
    while True:
        remaining = deadline - time.perf_counter()
        if remaining <= 0:
            return
        if remaining > spin_s:
            time.sleep(remaining - spin_s)
        else:
            while time.perf_counter() < deadline:
                pass
            return


def progress_bar(n_cycles, period_s, off_s, desc, enabled=True):
    """Live tqdm bar, already rendered - or None if it would disturb the timing.

    The bar is updated once per cycle, between the OFF edge and the next ON
    deadline, so a steady-state update is absorbed by the OFF dwell. The *first*
    render is the expensive one (tens of ms in Jupyter), so it is paid here,
    before the schedule starts, and the bar is dropped if even that is too slow
    for the dwell you asked for. You always get told why.
    """
    if not enabled:
        return None
    if period_s < BAR_MIN_PERIOD_S:
        print(f"note: progress bar off - {period_s*1e3:g} ms period is below "
              f"{BAR_MIN_PERIOD_S*1e3:g} ms, a per-cycle UI update would disturb the timing")
        return None
    try:
        from tqdm.auto import tqdm
    except ImportError:
        print("note: progress bar off - tqdm is not installed (uv add tqdm)")
        return None
    bar = tqdm(total=n_cycles, desc=desc, unit="cy")
    tic = time.perf_counter()
    try:
        bar.refresh()
    except Exception:
        bar.update(0)
    warm = time.perf_counter() - tic
    if warm > 0.25 * off_s:
        bar.close()
        print(f"note: progress bar off - first render took {warm*1e3:.0f} ms, "
              f"too slow for a {off_s*1e3:g} ms OFF dwell")
        return None
    return bar


@contextlib.contextmanager
def timing_boost(pin_cpu=False):
    """Best-effort low-jitter context: 1 ms Windows timer + higher priority.

    Both are restored on exit, whatever happens (including KeyboardInterrupt).
    Yields a status dict so you can see which optimisations really applied -
    silently swallowing failures is how you end up benchmarking nothing.
    """
    status = {"timer_1ms": False, "priority": "unchanged", "pinned": False}
    proc, old_nice, old_aff = None, None, None
    try:
        import psutil
        proc = psutil.Process()
        old_nice = proc.nice()
        if sys.platform == "win32":
            import ctypes
            ctypes.windll.winmm.timeBeginPeriod(1)
            status["timer_1ms"] = True
            proc.nice(psutil.HIGH_PRIORITY_CLASS)  # never REALTIME: it can starve the OS
            status["priority"] = "HIGH_PRIORITY_CLASS"
        else:
            proc.nice(max(old_nice - 10, -20))
            status["priority"] = f"nice {old_nice} -> {proc.nice()}"
        if pin_cpu:                                # optional, rarely worth the risk
            old_aff = proc.cpu_affinity()
            proc.cpu_affinity([0])
            status["pinned"] = True
    except Exception as exc:
        status["error"] = f"{type(exc).__name__}: {exc}"
    try:
        yield status
    finally:
        if status["timer_1ms"]:
            try:
                ctypes.windll.winmm.timeEndPeriod(1)
            except Exception:
                pass
        if proc is not None:
            try:
                proc.nice(old_nice)
                if old_aff is not None:
                    proc.cpu_affinity(old_aff)
            except Exception:
                pass

## 3  Connect and check the head

One connection for the whole notebook — opened here, closed in §7. Do not create a
second `CoboltLaser` on the same port: Windows refuses to open it twice.

In [ ]:
laser = pycobolt.CoboltLaser(port=COM_PORT)     # connects at once; SerialException if the port is busy
print("Connected:", laser)                      # serial / model / wavelength / type

if laser.interlock() != "0":                    # 0 = closed, 1 = open
    raise RuntimeError("Interlock is OPEN - close it before enabling emission")
fault = laser.get_fault()
if fault not in ("0", "", None):
    print("Fault:", fault, "-> clearing:", laser.clear_fault())
print("Mode :", laser.get_mode())
print("State:", laser.get_state())

laser.turn_off()                                # start from a known dark state
print(f"Head limit {MAX_POWER_MW:.0f} mW · default protocol {POWER_MW:.0f} mW, "
      f"{ON_S*1e3:g}/{OFF_S*1e3:g} ms, {N_CYCLES} cycles")

## 4  Routine

In [ ]:
def blink(laser, power_mw, on_s, off_s, n_cycles, spin_s=0.001, progress=True):
    """Toggle emission on an absolute monotonic schedule.

    Returns (rec, missed). `rec[phase]` is an (n_cycles, 2) array of seconds:
      col 0  scheduling error - how late the command was sent vs. its deadline
      col 1  command latency  - how long the serial call took to return
    Both are >= 0; cycles that never ran stay NaN. `missed` counts edges sent
    later than half the shorter dwell.

    Emission is always left OFF. The serial connection belongs to the caller and
    is left open, so the laser object stays reusable across runs.
    """
    if min(power_mw, on_s, off_s) <= 0:
        raise ValueError("power_mw, on_s and off_s must all be > 0")
    if n_cycles < 1:
        raise ValueError("n_cycles must be >= 1")

    period = on_s + off_s
    miss_threshold = min(on_s, off_s) / 2
    rec = {phase: np.full((n_cycles, 2), np.nan) for phase in ("on", "off")}
    missed = 0
    desc = f"{power_mw:g} mW · {on_s*1e3:g}/{off_s*1e3:g} ms"

    with timing_boost() as boost:
        print("timing boost:", boost)
        laser.constant_power(power_mw)          # two round-trips: "p <W>" then "cp"
        print("setpoint:", call_if_available(laser, "get_power_setpoint"), "mW")
        laser.turn_off()
        # Bar first: its initial render is the slow part and must not eat the lead-in.
        bar = progress_bar(n_cycles, period, off_s, desc, progress)
        # Deadlines are set AFTER these setup round-trips, which take ~10 ms each.
        next_on = time.perf_counter() + 0.05     # small lead-in
        next_off = next_on + on_s
        try:
            for k in range(n_cycles):
                for phase, deadline, command in (("on", next_on, laser.turn_on),
                                                 ("off", next_off, laser.turn_off)):
                    wait_until(deadline, spin_s)
                    sent = time.perf_counter()
                    command()
                    rec[phase][k] = (sent - deadline, time.perf_counter() - sent)
                    missed += (sent - deadline) > miss_threshold
                next_on += period
                next_off += period
                if bar is not None:
                    bar.update(1)                # between edges: absorbed by the OFF dwell
        finally:
            if bar is not None:
                bar.close()
            try:
                laser.turn_off()                 # runs on Ctrl-C and on errors too
            except Exception as exc:
                print(f"warning: final turn_off failed: {exc}")   # never mask the real error
    return rec, missed


def _ms(values):
    """Format one timing column as mean / p95 / max in ms."""
    v = values[np.isfinite(values)]
    if v.size == 0:
        return "no data"
    return (f"mean {v.mean()*1e3:6.2f} ms   p95 {np.percentile(v, 95)*1e3:6.2f} ms"
            f"   max {v.max()*1e3:6.2f} ms")


def report(rec, on_s, off_s, missed):
    print(f"{len(rec['on'])} cycles, {missed} missed deadline(s)")
    for phase, planned in (("on", on_s), ("off", off_s)):
        print(f"{phase.upper():3s} planned {planned*1e3:7.1f} ms")
        print(f"      scheduling error : {_ms(rec[phase][:, 0])}")
        print(f"      command latency  : {_ms(rec[phase][:, 1])}")


def save_csv(directory, rec, *, on_s, off_s, power_mw, port, laser, missed, spin_s, tag=None):
    """Write one timestamped CSV per run, with a '#' metadata header.

    Never overwrites: a second run in the same clock second gets a _1 suffix.
    Call while the laser is still connected: repr(laser) queries the head.
    """
    directory = pathlib.Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    stem = f"blink{('_' + tag) if tag else ''}_{stamp}"
    path, n = directory / f"{stem}.csv", 1
    while path.exists():
        path = directory / f"{stem}_{n}.csv"
        n += 1

    meta = [("timestamp", datetime.now().isoformat(timespec="seconds")),
            ("port", port), ("laser", " ".join(str(laser).split())),
            ("power_mw", power_mw), ("on_s", on_s), ("off_s", off_s),
            ("cycles", len(rec["on"])), ("missed", missed), ("spin_s", spin_s),
            ("pycobolt", pkg_version("pycobolt")),
            ("python", sys.version.split()[0]), ("platform", sys.platform)]
    with path.open("w", newline="") as fh:
        for key, value in meta:
            fh.write(f"# {key}={value}\n")
        writer = csv.writer(fh)
        writer.writerow(["cycle", "phase", "planned_s", "late_s", "cmd_s"])
        for k in range(len(rec["on"])):
            for phase, planned in (("on", on_s), ("off", off_s)):
                late, cmd = rec[phase][k]
                writer.writerow([k, phase, planned, f"{late:.9f}", f"{cmd:.9f}"])
    print("saved:", path)
    return path


def plot_timing(rec, power_mw, on_s, off_s):
    """Per-cycle scheduling error (top) and command latency (bottom)."""
    idx = np.arange(len(rec["on"]))
    fig, axes = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    for ax, col, label in ((axes[0], 0, "scheduling error (s)"),
                           (axes[1], 1, "command latency (s)")):
        for phase, marker in (("on", "o"), ("off", "x")):
            ax.plot(idx, rec[phase][:, col], marker, ls="none", ms=4, label=phase)
        ax.axhline(0, ls="--", lw=0.8)
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
        ax.legend(loc="upper right", fontsize=8)
    axes[0].set_title(f"Cobolt {power_mw:g} mW · {on_s*1e3:g}/{off_s*1e3:g} ms · {len(idx)} cycles")
    axes[1].set_xlabel("cycle #")
    fig.tight_layout()
    return fig

In [ ]:
RUNS = []   # one entry per completed run; see summary_table()


def run(power_mw=None, on_s=None, off_s=None, n_cycles=None, *, tag=None,
        spin_s=None, progress=PROGRESS, plot=True):
    """One complete run: clamp power, blink, report, save CSV, plot, remember it.

    Any argument left as None falls back to the §1 defaults, read at call time -
    so a run cell only has to name what changes. Re-run it as often as you like:
    each call writes its own timestamped CSV and appends a row to RUNS.
    Returns (rec, missed, csv_path).
    """
    power_mw = min(POWER_MW if power_mw is None else power_mw, MAX_POWER_MW)
    on_s     = ON_S     if on_s     is None else on_s
    off_s    = OFF_S    if off_s    is None else off_s
    n_cycles = N_CYCLES if n_cycles is None else n_cycles
    spin_s   = SPIN_S   if spin_s   is None else spin_s

    if not laser.is_connected():
        raise RuntimeError("laser is not connected - re-run §3")

    label = f"Run {len(RUNS) + 1}" + (f" [{tag}]" if tag else "")
    print(f"\n=== {label}: {power_mw:g} mW, {on_s*1e3:g}/{off_s*1e3:g} ms, "
          f"{n_cycles} cycles (~{n_cycles * (on_s + off_s):g} s) ===")

    rec, missed = blink(laser, power_mw, on_s, off_s, n_cycles,
                        spin_s=spin_s, progress=progress)
    report(rec, on_s, off_s, missed)
    path = save_csv(LOG_DIR, rec, on_s=on_s, off_s=off_s, power_mw=power_mw,
                    port=COM_PORT, laser=laser, missed=missed, spin_s=spin_s, tag=tag)
    if plot:
        plot_timing(rec, power_mw, on_s, off_s)

    RUNS.append({"n": len(RUNS) + 1, "tag": tag or "", "power_mw": power_mw,
                 "on_s": on_s, "off_s": off_s, "n_cycles": n_cycles, "missed": missed,
                 "late_p95": np.nanpercentile(rec["on"][:, 0], 95),
                 "cmd_mean": np.nanmean(rec["on"][:, 1]),
                 "rec": rec, "csv": str(path)})
    return rec, missed, path


def summary_table():
    """One row per completed run - compare powers and dwell times at a glance."""
    if not RUNS:
        print("no runs yet - see §5")
        return None
    cols = ("n", "tag", "power_mw", "on_s", "off_s", "n_cycles", "missed", "late_p95", "cmd_mean")
    df = pd.DataFrame([{k: r[k] for k in cols} for r in RUNS])
    for c in ("late_p95", "cmd_mean"):
        df[f"{c}_ms"] = (df.pop(c) * 1e3).round(2)
    return df


def replot(n=1):
    """Re-plot an earlier run from memory, e.g. replot(2)."""
    r = next(x for x in RUNS if x["n"] == n)
    print(f"run {r['n']} -> {r['csv']}")
    return plot_timing(r["rec"], r["power_mw"], r["on_s"], r["off_s"])

## 5  Runs

Edit a cell and run it — repeatedly, in any order, with different powers and dwell
times. Anything you omit comes from §1. Each run gets its own CSV in `logs/` and a
row in the §6 table. `tag=` ends up in the filename, so name protocols usefully.

In [ ]:
# Run 1 - gentle: 20 mW, 1 s on / 1 s off
run(power_mw=20, on_s=1.0, off_s=1.0, n_cycles=10, tag="20mW_1s")

In [ ]:
# Run 2 - fast blinking at full rated power (50 ms pulses)
run(power_mw=100, on_s=0.05, off_s=0.05, n_cycles=200, tag="100mW_50ms")

In [ ]:
# Run 3 - copy this cell for each new protocol; §1 defaults fill the gaps
# run(on_s=0.2, off_s=0.2, n_cycles=50, tag="200ms")

In [ ]:
# Power sweep - one run per level, all logged and compared in §6.
# Uncomment when the sample is in place and you are ready for N x the exposure.
# for mw in (20, 40, 60, 80, 100):
#     run(power_mw=mw, on_s=1.0, off_s=1.0, n_cycles=5, tag=f"sweep_{mw}mW")

## 6  Compare runs

In [ ]:
summary_table()

## 7  Disconnect

## Caveats
- **Software timings only.** `late_s` and `cmd_s` say nothing about the optical
  pulse. ~10 ms of serial latency on a 50 ms pulse is ~20 % error — if that
  matters, move to the modulation input, not to a tighter Python loop.
- **Faster than serial toggling.** Your head is a `Cobolt06MLD`, which pycobolt
  drives with `modulation_mode()`, `digital_modulation(1)`, `analog_modulation(1)`,
  `on_off_modulation(1)`, `set_modulation_power()` and `set_analog_impedance()`.
  Enable digital/analogue modulation once and feed the modulation input from a
  function generator or DAQ: hardware timing, no ~10 ms serial round-trip per edge.
- **`missed` is the number to watch.** Absolute scheduling keeps the duty cycle
  but never re-syncs, so one slow command shifts the phase of everything after it.
- **Legacy CSVs are not comparable.** The old files (`test.csv`, `m15*.csv`,
  `multiply_*.csv`) use an `actual_s` column from an earlier metric — some even
  contain negative values, which the current definition cannot produce. Treat them
  as a different measurement, or re-run the sweeps with this notebook.

In [ ]:
laser.disconnect()   # the only place the port is released